# NLP Demo: Subspace Surgery on Tiny Shakespeare

This notebook reproduces the headline results from the paper:

- Train a full-A linear SSM on a 3-channel EMA task over real text (Tiny Shakespeare)
- Extract balanced circuits via `ssm_interpret`
- Ablate each circuit and measure channel-selective degradation
- Compare to activation patching (non-selective baseline)
- Confirm the Glover bound holds for the truncated model

**Expected results (from §3 of the paper):**
- Surgery selectivity: ~251× for the slow EMA circuit (mode k=0)
- Faithfulness ratio: ~12.3× (active vs inactive sequences)
- Activation patching selectivity: ~1.5× (non-selective, as expected)

**Task:** For each character in Tiny Shakespeare, compute 3 binary features:
- `is_alpha` (ρ=0.95, slow circuit)
- `is_space` (ρ=0.65, medium circuit)
- `is_upper` (ρ=0.20, fast circuit)

The SSM learns to predict the EMA of each feature at its assigned timescale.

In [ ]:
import subprocess, sys
# Install ssm-interpret if not already installed
try:
    import ssm_interpret
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '..'])

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings

import ssm_interpret as ssi

print(f'ssm_interpret version: {ssi.__version__}')
print(f'torch version: {torch.__version__}')

## 1. Corpus and Feature Extraction

In [ ]:
import os, subprocess as sp

CORPUS_URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
LOCAL_CACHE = '/tmp/tinyshakespeare.txt'

FALLBACK = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
MACBETH: Tomorrow, and tomorrow, and tomorrow,
Creeps in this petty pace from day to day,
KING LEAR: Blow, winds, and crack your cheeks! rage! blow!
OTHELLO: She loved me for the dangers I had pass'd,
And I loved her that she did pity them.
""" * 200

def load_corpus():
    if os.path.exists(LOCAL_CACHE):
        with open(LOCAL_CACHE) as f:
            corpus = f.read()
        print(f'Corpus: local cache ({len(corpus):,} chars)')
        return corpus
    try:
        out = sp.check_output(['curl', '-s', '--max-time', '15', CORPUS_URL], timeout=20)
        corpus = out.decode('utf-8')
        if len(corpus) > 10000:
            with open(LOCAL_CACHE, 'w') as f:
                f.write(corpus)
            print(f'Corpus: downloaded ({len(corpus):,} chars)')
            return corpus
    except Exception:
        pass
    print(f'Corpus: fallback ({len(FALLBACK):,} chars)')
    return FALLBACK

corpus = load_corpus()

# Feature extraction
M = 3
features = np.zeros((len(corpus), M), dtype=np.float32)
for i, c in enumerate(corpus):
    features[i, 0] = float(c.isalpha())
    features[i, 1] = float(c in ' \t\n')
    features[i, 2] = float(c.isupper())

RHOS = [0.95, 0.65, 0.20]
FEAT_LABELS = ['is_alpha (ρ=0.95, slow)', 'is_space (ρ=0.65, med)', 'is_upper (ρ=0.20, fast)']

# EMA targets
targets = np.zeros_like(features)
for k, rho in enumerate(RHOS):
    state = 0.0
    for t in range(len(corpus)):
        state = rho * state + (1 - rho) * features[t, k]
        targets[t, k] = state

print(f'Feature means: {features.mean(0).round(3)}')
print(f'Target means:  {targets.mean(0).round(3)}')

In [ ]:
# Build sequence dataset
T_SEQ = 64
N_TR, N_TEST = 3000, 512
stride = T_SEQ // 2

starts = list(range(0, len(features) - T_SEQ, stride))
seqs_f = np.stack([features[s:s+T_SEQ] for s in starts])
seqs_t = np.stack([targets[s:s+T_SEQ] for s in starts])

rng = np.random.default_rng(0)
idx = rng.permutation(len(seqs_f))
tr_idx, te_idx = idx[:N_TR], idx[N_TR:N_TR+N_TEST]

u_tr = torch.tensor(seqs_f[tr_idx])
y_tr = torch.tensor(seqs_t[tr_idx])
u_te = torch.tensor(seqs_f[te_idx])
y_te = torch.tensor(seqs_t[te_idx])

null_mse = y_te.pow(2).mean(dim=(0, 1)).numpy()
print(f'Dataset: {len(tr_idx)} train, {len(te_idx)} test sequences of length {T_SEQ}')
print(f'Null MSE per channel: {null_mse.round(4)}')

## 2. Model Definition and Training

In [ ]:
D_STATE = 3  # exact capacity: one circuit per EMA channel
EPOCHS = 25000
LR = 3e-3

class LinearSSM(nn.Module):
    def __init__(self, m, d, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        self.m, self.d = m, d
        self.A_raw = nn.Parameter(torch.randn(d, d) * 0.1)
        self.B = nn.Parameter(torch.randn(d, m) * 0.1)
        self.C = nn.Parameter(torch.randn(m, d) * 0.1)

    def _A(self):
        sr = torch.linalg.matrix_norm(self.A_raw, ord=2)
        return self.A_raw / sr.clamp(min=1.0) * 0.99

    def forward(self, u):
        B_, T_, _ = u.shape
        A = self._A()
        x = torch.zeros(B_, self.d)
        ys = []
        for t in range(T_):
            x = x @ A.T + u[:, t, :] @ self.B.T
            ys.append(x @ self.C.T)
        return torch.stack(ys, dim=1)

    def get_matrices(self):
        with torch.no_grad():
            return self._A().numpy().astype(np.float64), \
                   self.B.detach().numpy().astype(np.float64), \
                   self.C.detach().numpy().astype(np.float64)

def train_model(model, u_tr, y_tr, epochs=EPOCHS, lr=LR, verbose=True):
    opt = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.MultiStepLR(opt, milestones=[8000, 16000], gamma=0.3)
    for ep in range(epochs):
        opt.zero_grad()
        F.mse_loss(model(u_tr), y_tr).backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        sched.step()
        if verbose and (ep + 1) % 5000 == 0:
            with torch.no_grad():
                loss = F.mse_loss(model(u_tr), y_tr).item()
            print(f'  epoch {ep+1:5d}  loss={loss:.4e}')
    with torch.no_grad():
        return F.mse_loss(model(u_tr), y_tr).item()

print(f'Model: LinearSSM d={D_STATE}, m={M} | Training for {EPOCHS} epochs...')
model = LinearSSM(M, D_STATE, seed=0)
final_loss = train_model(model, u_tr, y_tr)
print(f'Final train loss: {final_loss:.4e}')

In [ ]:
# Baseline test performance
with torch.no_grad():
    y_base = model(u_te).numpy()

mse_base = np.mean((y_base - y_te.numpy())**2, axis=(0, 1))
pct_base = 100 * mse_base / (null_mse + 1e-12)
print('Baseline test MSE% per channel:')
for k, (label, pct) in enumerate(zip(FEAT_LABELS, pct_base)):
    print(f'  ch{k} {label}: {pct:.2f}%')

## 3. Balanced Realization: HSV Spectrum

In [ ]:
A, B, C = model.get_matrices()

Ab, Bb, Cb, T_bal, sigma = ssi.balanced_realization(A, B, C, T=T_SEQ)
print(f'Hankel singular values: {sigma.round(4)}')
print(f'HSV ratio sigma[0]/sigma[-1]: {sigma[0]/sigma[-1]:.1f}×')

# Glover bound for k=1 (keep top-1 mode)
bound_k1 = ssi.glover_bound(sigma, k=1)
print(f'Glover bound (keep k=1 mode): ||H - H_1||_H <= {bound_k1:.4f}')

# C_bal energy: which channel does each mode serve?
Cb_np = np.array(Cb)
c_energy = Cb_np**2  # (M, d)
c_frac = c_energy / (c_energy.sum(axis=0, keepdims=True) + 1e-12)

print('\nC_bal energy fraction (rows=channel, cols=mode):')
print('  ' + '  '.join(f'mode {k}' for k in range(D_STATE)))
for ch, label in enumerate(FEAT_LABELS):
    row = '  '.join(f'{c_frac[ch, k]:.3f} ' for k in range(D_STATE))
    print(f'  ch{ch}: {row}  | {label}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# HSV spectrum
ax = axes[0]
ax.bar(range(D_STATE), sigma, color=['#2196F3', '#4CAF50', '#FF9800'], edgecolor='k', linewidth=0.8)
ax.set_xlabel('Mode index k')
ax.set_ylabel('Hankel singular value σ_k')
ax.set_title('HSV Spectrum')
ax.set_xticks(range(D_STATE))
for k, s in enumerate(sigma):
    ax.text(k, s + 0.01, f'{s:.3f}', ha='center', va='bottom', fontsize=10)

# C_bal energy heatmap
ax = axes[1]
im = ax.imshow(c_frac[:, :D_STATE], cmap='Blues', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Energy fraction')
ax.set_xlabel('Mode index k')
ax.set_ylabel('Output channel')
ax.set_title('C_bal Energy Allocation\n(mode k → channel)')
ax.set_xticks(range(D_STATE))
ax.set_yticks(range(M))
ax.set_yticklabels([f'ch{i}' for i in range(M)])
for i in range(M):
    for j in range(D_STATE):
        ax.text(j, i, f'{c_frac[i,j]:.2f}', ha='center', va='center',
                color='white' if c_frac[i,j] > 0.5 else 'black', fontsize=10)

plt.tight_layout()
plt.savefig('hsv_spectrum.pdf', bbox_inches='tight')
plt.show()
print('Fig saved: hsv_spectrum.pdf')

## 4. Subspace Surgery: Circuit-Selective Ablation

For each balanced mode k, we ablate it using `ssi.ablate(mode='output')` and measure how much each output channel degrades. A highly selective ablation damages only the target channel.

In [ ]:
def run_ssm(A, B, C, u):
    """Simulate SSM on input batch u: (B, T, m) -> (B, T, p)."""
    A_t = torch.tensor(A, dtype=torch.float32)
    B_t = torch.tensor(B, dtype=torch.float32)
    C_t = torch.tensor(C, dtype=torch.float32)
    Bsz, T_, _ = u.shape
    x = torch.zeros(Bsz, A.shape[0])
    ys = []
    with torch.no_grad():
        for t in range(T_):
            x = x @ A_t.T + u[:, t, :] @ B_t.T
            ys.append(x @ C_t.T)
    return torch.stack(ys, dim=1).numpy()

print('Running surgery for each mode k...')
sel_ratios = []
mode_targets = []

for k in range(D_STATE):
    A_s, B_s, C_s = ssi.ablate(A, B, C, T_bal, k=k, mode='output')
    y_surg = run_ssm(np.array(A_s), np.array(B_s), np.array(C_s), u_te)

    # Null model: predict zeros
    y_null = np.zeros_like(y_base)

    sel = ssi.selectivity(
        y_surg, y_base, y_null,
        target_channel=int(np.argmax(c_frac[:, k]))
    )
    target_ch = int(np.argmax(c_frac[:, k]))
    mode_targets.append(target_ch)
    sel_ratios.append(sel)

    mse_s = np.mean((y_surg - y_base)**2, axis=(0, 1))
    delta = mse_s / (null_mse + 1e-12)
    print(f'  mode k={k} (σ={sigma[k]:.4f}) → targets ch{target_ch} ({FEAT_LABELS[target_ch][:18]})')
    print(f'    Δ = {delta.round(3)}  selectivity = {sel:.1f}×')

In [ ]:
# Activation patching comparison: zero state at T/2
t0 = T_SEQ // 2
A_t = torch.tensor(A, dtype=torch.float32)
B_t = torch.tensor(B, dtype=torch.float32)
C_t = torch.tensor(C, dtype=torch.float32)
Bsz = u_te.shape[0]
x = torch.zeros(Bsz, D_STATE)
ys_patch = []
with torch.no_grad():
    for t in range(T_SEQ):
        if t == t0:
            x = torch.zeros_like(x)
        x = x @ A_t.T + u_te[:, t, :] @ B_t.T
        ys_patch.append(x @ C_t.T)
y_patch = torch.stack(ys_patch, dim=1).numpy()

mse_patch = np.mean((y_patch - y_base)**2, axis=(0, 1))
delta_patch = mse_patch / (null_mse + 1e-12)
patch_sel = delta_patch.max() / (delta_patch.mean() + 1e-6)
print(f'Activation patching (zero state at t={t0}):')
print(f'  Δ = {delta_patch.round(3)}  max-selectivity = {patch_sel:.2f}×')
print(f'\nSurgery vs patching gap: {np.mean(sel_ratios)/patch_sel:.0f}× improvement in selectivity')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800']

# Selectivity ratios
ax = axes[0]
bars = ax.bar(range(D_STATE), sel_ratios, color=[colors[t] for t in mode_targets],
              edgecolor='k', linewidth=0.8)
ax.axhline(y=patch_sel, color='gray', linestyle='--', linewidth=1.5, label=f'Patching ({patch_sel:.1f}×)')
ax.axhline(y=1.0, color='black', linestyle=':', linewidth=1, alpha=0.5, label='Random baseline (1×)')
ax.set_xlabel('Ablated mode k')
ax.set_ylabel('Selectivity ratio')
ax.set_title('Surgery Selectivity')
ax.set_xticks(range(D_STATE))
ax.set_xticklabels([f'k={k}\n(ch{mode_targets[k]})' for k in range(D_STATE)])
ax.legend()
for k, s in enumerate(sel_ratios):
    ax.text(k, s + max(sel_ratios)*0.02, f'{s:.0f}×', ha='center', va='bottom', fontweight='bold')

# Per-channel damage heatmap
ax = axes[1]
delta_matrix = np.zeros((D_STATE, M))
for k in range(D_STATE):
    A_s, B_s, C_s = ssi.ablate(A, B, C, T_bal, k=k, mode='output')
    y_s = run_ssm(np.array(A_s), np.array(B_s), np.array(C_s), u_te)
    mse_s = np.mean((y_s - y_base)**2, axis=(0, 1))
    delta_matrix[k] = mse_s / (null_mse + 1e-12)

im = ax.imshow(delta_matrix, cmap='Reds', aspect='auto')
plt.colorbar(im, ax=ax, label='Normalised damage Δ')
ax.set_xlabel('Output channel')
ax.set_ylabel('Ablated mode k')
ax.set_title('Per-Channel Damage\n(high Δ on diagonal = selective)')
ax.set_xticks(range(M))
ax.set_xticklabels([f'ch{i}' for i in range(M)])
ax.set_yticks(range(D_STATE))
ax.set_yticklabels([f'mode {k}' for k in range(D_STATE)])
for i in range(D_STATE):
    for j in range(M):
        ax.text(j, i, f'{delta_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('surgery_selectivity.pdf', bbox_inches='tight')
plt.show()
print('Fig saved: surgery_selectivity.pdf')

## 5. Faithfulness: Input-Conditional Causal Validity

In [ ]:
# Mode k=0 targets is_alpha (slow circuit).
# Active sequences: high variance in is_alpha (lots of alpha chars)
# Inactive sequences: low variance (e.g. mostly spaces)

# Measure per-sequence is_alpha activity
alpha_activity = u_te[:, :, 0].numpy().std(axis=1)  # (N_TEST,)
med = np.median(alpha_activity)

active_mask = alpha_activity > med
inactive_mask = ~active_mask
print(f'Active sequences (high is_alpha variance): {active_mask.sum()}')
print(f'Inactive sequences: {inactive_mask.sum()}')

# Ablate mode k=0
A_s0, B_s0, C_s0 = ssi.ablate(A, B, C, T_bal, k=0, mode='output')
y_surg0 = run_ssm(np.array(A_s0), np.array(B_s0), np.array(C_s0), u_te)

# Delta (damage) on active vs inactive, for the target channel (ch0 = is_alpha)
target_ch = mode_targets[0]  # should be ch0
diff_active = np.mean((y_surg0[active_mask, :, target_ch] - y_base[active_mask, :, target_ch])**2)
diff_inactive = np.mean((y_surg0[inactive_mask, :, target_ch] - y_base[inactive_mask, :, target_ch])**2)
null_active = np.mean(y_te.numpy()[active_mask, :, target_ch]**2)
null_inactive = np.mean(y_te.numpy()[inactive_mask, :, target_ch]**2)

delta_active = diff_active / (null_active + 1e-12)
delta_inactive = diff_inactive / (null_inactive + 1e-12)
faith = ssi.faithfulness_ratio(delta_active, delta_inactive)

print(f'\nFaithfulness analysis (mode k=0, target ch{target_ch}):')
print(f'  Delta (active seqs):   {delta_active:.4f}')
print(f'  Delta (inactive seqs): {delta_inactive:.4f}')
print(f'  Faithfulness ratio:    {faith:.1f}×')
print(f'  Interpretation: ablating the slow circuit is {faith:.0f}× more damaging')
print(f'  when the relevant feature (is_alpha) is present in the input.')

## 6. Multi-Seed Summary

In [ ]:
N_SEEDS = 5
all_sel = []

print('Running multi-seed experiment...')
for seed in range(N_SEEDS):
    m = LinearSSM(M, D_STATE, seed=seed)
    train_model(m, u_tr, y_tr, verbose=False)

    with torch.no_grad():
        y_b = m(u_te).numpy()

    A_, B_, C_ = m.get_matrices()
    try:
        _, _, Cb_, T_bal_, sigma_ = ssi.balanced_realization(A_, B_, C_, T=T_SEQ)
        Cb_np_ = np.array(Cb_)
        c_frac_ = (Cb_np_**2) / ((Cb_np_**2).sum(axis=0, keepdims=True) + 1e-12)
        seed_sel = []
        for k in range(D_STATE):
            A_s_, B_s_, C_s_ = ssi.ablate(A_, B_, C_, T_bal_, k=k, mode='output')
            y_s_ = run_ssm(np.array(A_s_), np.array(B_s_), np.array(C_s_), u_te)
            null_ = y_te.pow(2).mean(dim=(0,1)).numpy()
            mse_s_ = np.mean((y_s_ - y_b)**2, axis=(0,1))
            delta_ = mse_s_ / (null_ + 1e-12)
            t_ch = int(np.argmax(c_frac_[:, k]))
            others_ = [float(delta_[j]) for j in range(M) if j != t_ch]
            sel_ = float(delta_[t_ch]) / (sum(others_)/len(others_) + 1e-6)
            seed_sel.append(sel_)
        all_sel.append(seed_sel)
        print(f'  seed={seed}: sel = {[f"{s:.0f}x" for s in seed_sel]}')
    except Exception as e:
        print(f'  seed={seed}: skipped ({e})')

if all_sel:
    sel_arr = np.array(all_sel)
    print(f'\nMean selectivity per mode: {sel_arr.mean(axis=0).round(1)}')
    print(f'Std  selectivity per mode: {sel_arr.std(axis=0).round(1)}')
    print(f'Grand mean: {sel_arr.mean():.1f}×')

## 7. Summary

| Metric | Value |
|--------|-------|
| Surgery selectivity (mode k=0, slow circuit) | ~251× |
| Faithfulness ratio (active vs inactive seqs) | ~12× |
| Activation patching selectivity | ~1.5× |
| Surgery vs patching gap | ~160× |

**Key takeaways:**
1. Balanced realization assigns one circuit per EMA timescale — the C_bal energy heatmap shows near-diagonal structure.
2. Subspace surgery is highly selective: ablating mode k damages primarily the EMA channel it serves.
3. Activation patching is non-selective: zeroing the state damages all channels equally because all circuits run through the same state vector.
4. Faithfulness confirms the circuit is causally active only when the relevant input feature is present.